In [ ]:
#| default_exp magickey

# MagicKey
> Passwordless auth combining magic links and passkeys

In [ ]:
#| export
import secrets, time
from json import loads
from urllib.parse import urlparse

from webauthn import generate_authentication_options, generate_registration_options, verify_authentication_response, verify_registration_response, options_to_json
from webauthn.helpers import base64url_to_bytes, bytes_to_base64url
from webauthn.helpers.structs import AuthenticatorSelectionCriteria, ResidentKeyRequirement, UserVerificationRequirement

from fastcore.utils import *
from fastcore.xml import *
from fastlite import *
from fasthtml.basics import *
from fasthtml.starlette import *
from fasthtml.fastapp import *

In [ ]:
from fastcore.test import *
from starlette.testclient import TestClient

## Helpers

`public_origin` and `rp_id` are passed to `MagicKey.__init__` and used directly at both WebAuthn verification call sites — no helper needed.


In [ ]:
#| export
# _origin_kw removed; public_origin / rp_id inlined at call sites


`_webauthn_js` builds an inline `<script>` that calls SimpleWebAuthn's browser API (either `startRegistration` or `startAuthentication`) with the given options, then POSTs the result back to the server via htmx.

In [ ]:
#| export
def _webauthn_js(opts, action, callback):
    return Script("SimpleWebAuthnBrowser.%s({ optionsJSON: %s }).then(r => {"
        "htmx.ajax('POST', '%s', { values: r });});" % (action, options_to_json(opts), callback))

## MagicKey class

Follows the same pattern as `OAuth`: wires up beforeware and backend routes in `__init__`, with overridable methods for app-specific behaviour.

The `MagicKey` class sets up beforeware (redirects unauthenticated users to the login page) and registers all backend routes. The developer subclasses it and overrides `get_auth` (what URL to redirect to after login) and the passkey storage methods.

In [ ]:
#| export
class MagicKey:
    def __init__(self,
                 app,             # FastHTML app instance
                 send_email,      # `f(email, magic_url)` — sends the magic link
                 public_origin,   # Full origin URL, e.g. `https://example.com`
                 rp_id=None,      # WebAuthn relying party ID (default: hostname from `public_origin`)
                 rp_name=None,    # WebAuthn relying party display name (default: `rp_id`)
                 skip=None,       # Routes to skip auth beforeware (default: all MagicKey routes)
                 login_path='/login',   # Login page route
                 logout_path='/logout', # Logout route
                 token_expiry=3600,    # Magic link validity in seconds
                 webauthn_js=None):    # Script tag or URL for SimpleWebAuthn browser JS (default: jsdelivr CDN)
        "Passwordless auth combining magic links and passkeys"
        if not rp_id: rp_id = urlparse(public_origin).hostname
        self.magiclink_db = {}
        store_attr()
        async def _before(req, session):
            if 'auth' not in req.scope: req.scope['auth'] = session.get('auth')
            if not req.scope['auth']: return RedirectResponse(login_path, status_code=303)
        if not skip: skip = [login_path, logout_path, '/finish_passkey_reg', '/send_magic_link', 
                             '/request_passkey_auth', '/request_passkey_reg', '/setup_passkey',
                             '/skip_passkey_reg', '/verify_magiclink', '/verify_passkey_auth',]
        app.before.append(Beforeware(_before, skip=skip))
        if not webauthn_js: webauthn_js = Script(src='https://cdn.jsdelivr.net/npm/@simplewebauthn/browser@13.1.0/dist/bundle/index.umd.min.js')
        elif isinstance(webauthn_js, str): webauthn_js = Script(src=webauthn_js)
        app.hdrs += (webauthn_js,)
        app.get(logout_path)(self._logout)
        app.post('/send_magic_link')(self._send_magic_link)
        app.get('/verify_magiclink')(self._verify_magiclink)
        app.post('/request_passkey_auth')(self._request_passkey_auth)
        app.post('/verify_passkey_auth')(self._verify_passkey_auth)
        app.post('/request_passkey_reg')(self._request_passkey_reg)
        app.post('/finish_passkey_reg')(self._finish_passkey_reg)
        app.post('/skip_passkey_reg')(self._skip_passkey_reg)

    def _logout(self, session):
        session.pop('auth', None)
        return RedirectResponse(self.login_path, status_code=303)

    def _send_magic_link(self, email: str):
        self.magiclink_db = {k:v for k,v in self.magiclink_db.items() if not v['used'] and time.time() - v['timestamp'] <= self.token_expiry}
        token = secrets.token_urlsafe(32)
        self.magiclink_db[token] = dict(email=email, timestamp=time.time(), used=False)
        magic_url = f'{self.public_origin}/verify_magiclink?token={token}'
        return self.send_email(email, magic_url)

    def _verify_magiclink(self, token: str, session, req):
        link = self.magiclink_db.get(token)
        if not link or link['used'] or time.time() - link['timestamp'] > self.token_expiry:
            return RedirectResponse(f'{self.login_path}?error=invalid_link', status_code=303)
        link['used'] = True
        return self.after_magiclink_verify(link['email'], session, req)

    def _request_passkey_auth(self, session):
        auth_opts = generate_authentication_options(
            rp_id=self.rp_id, user_verification=UserVerificationRequirement.REQUIRED)
        session['auth_challenge'] = bytes_to_base64url(auth_opts.challenge)
        return _webauthn_js(auth_opts, 'startAuthentication', '/verify_passkey_auth')

    def _verify_passkey_auth(self, response: str, id: str, rawId: str, type: str, session):
        challenge_b64 = session.pop('auth_challenge', None)
        if not challenge_b64: return HttpHeader('HX-Redirect', f'{self.login_path}?error=no_challenge')
        stored = self.get_passkey(id)
        if not stored: return HttpHeader('HX-Redirect', f'{self.login_path}?error=passkey_not_found')
        try:
            res = verify_authentication_response(
                credential=dict(id=id, rawId=rawId, response=loads(response), type=type),
                credential_public_key=stored['public_key'], 
                credential_current_sign_count=stored['sign_count'],
                expected_challenge=base64url_to_bytes(challenge_b64), 
                require_user_verification=True,
                expected_origin=self.public_origin, expected_rp_id=self.rp_id)
        except Exception: return HttpHeader('HX-Redirect', f'{self.login_path}?error=passkey_failed')
        self.update_passkey(id, res.new_sign_count)
        session['auth'] = self.get_user_id(stored['email'])
        return self._do_auth(session['auth'], session, htmx=True)

    def _request_passkey_reg(self, session):
        email = session.get('pending_email')
        if not email: return RedirectResponse(f'{self.login_path}?error=no_pending_reg', status_code=303)
        opts = generate_registration_options(
            rp_id=self.rp_id, rp_name=self.rp_name or self.rp_id, user_name=email, 
            user_id=str(self.get_user_id(email)).encode(), 
            authenticator_selection=AuthenticatorSelectionCriteria(resident_key=ResidentKeyRequirement.REQUIRED))
        session['reg_challenge'] = bytes_to_base64url(opts.challenge)
        return _webauthn_js(opts, 'startRegistration', '/finish_passkey_reg')

    def _finish_passkey_reg(self, response: str, id: str, rawId: str, type: str, session):
        email = session.pop('pending_email', None)
        if not email: return RedirectResponse(f'{self.login_path}?error=no_pending_reg', status_code=303)
        challenge_b64 = session.pop('reg_challenge', None)
        if not challenge_b64: return RedirectResponse('/setup_passkey?error=no_challenge', status_code=303)
        try: res = verify_registration_response(credential=dict(id=id, rawId=rawId, response=loads(response), type=type),
                                                expected_challenge=base64url_to_bytes(challenge_b64),
                                                require_user_verification=True,
                                                expected_origin=self.public_origin, expected_rp_id=self.rp_id)
        except Exception: return RedirectResponse('/setup_passkey?error=reg_failed', status_code=303)
        self.save_passkey(bytes_to_base64url(res.credential_id), email, res.credential_public_key, res.sign_count)
        session['auth'] = self.get_user_id(email)
        return self._do_auth(session['auth'], session, htmx=True)

    def _skip_passkey_reg(self, session):
        email = session.pop('pending_email', None)
        if not email: return RedirectResponse(self.login_path, status_code=303)
        session['auth'] = self.get_user_id(email)
        return self._do_auth(session['auth'], session, htmx=False)

    def _do_auth(self, user_id, session, htmx=False):
        url = self.get_auth(user_id, session)
        if htmx: return HttpHeader('HX-Redirect', url)
        return RedirectResponse(url, status_code=303)

    def get_auth(self, user_id, session): return '/'
    def get_user_id(self, email): raise NotImplementedError()
    def has_passkey(self, email): raise NotImplementedError()
    def get_passkey(self, credential_id): raise NotImplementedError()
    def save_passkey(self, credential_id, email, public_key, sign_count): raise NotImplementedError()
    def update_passkey(self, credential_id, sign_count): raise NotImplementedError()

    def after_magiclink_verify(self, email, session, req):
        if self.has_passkey(email):
            session['auth'] = self.get_user_id(email)
            return self._do_auth(session['auth'], session, htmx=False)
        session['pending_email'] = email
        return RedirectResponse('/setup_passkey', status_code=303)

### Overridable methods

| Method | Purpose | Default |
|--------|---------|---------|
| `get_auth(user_id, session)` | Return URL to redirect to after successful auth | `/` |
| `get_user_id(email)` | Return user ID for a given email | `NotImplementedError` |
| `has_passkey(email)` | Check if user has a registered passkey | `NotImplementedError` |
| `get_passkey(credential_id)` | Look up passkey by credential ID, return dict with `email`, `public_key`, `sign_count` | `NotImplementedError` |
| `save_passkey(credential_id, email, public_key, sign_count)` | Store a new passkey | `NotImplementedError` |
| `update_passkey(credential_id, sign_count)` | Update sign count after auth | `NotImplementedError` |
| `after_magiclink_verify(email, session, req)` | Called after magic link verified; default offers passkey registration | redirects to `/setup_passkey` |

## Tests -

We can test the non-WebAuthn parts using FastHTML's test client. The beforeware, magic link flow, and session management are all testable without a browser.

In [ ]:
db = database(':memory:')
class User: id:int; email:str
class Passkey: id:str; user_id:int; public_key:bytes; sign_count:int
users = db.create(User)
passkeys = db.create(Passkey)

In [ ]:
class Auth(MagicKey):
    def get_user_id(self, email):
        res = users(where="email = ?", where_args=[email])
        if res: return res[0].id
        return users.insert(User(email=email)).id
    def has_passkey(self, email):
        return bool(passkeys(where="user_id = ?", where_args=[self.get_user_id(email)]))
    def get_passkey(self, cred_id):
        try: r = passkeys[cred_id]
        except NotFoundError: return None
        return dict(email=users[r.user_id].email, public_key=r.public_key, sign_count=r.sign_count)
    def save_passkey(self, cred_id, email, public_key, sign_count):
        uid = self.get_user_id(email)
        passkeys.insert(Passkey(id=cred_id, user_id=uid, public_key=public_key, sign_count=sign_count))
    def update_passkey(self, cred_id, sign_count):
        passkeys.update(Passkey(id=cred_id, sign_count=sign_count))

In [ ]:
def fake_send_email(email, url): return P(f'Magic link for {email}: ', A(url, href=url))

In [ ]:
app, rt = fast_app()
mk = Auth(app, send_email=fake_send_email, public_origin='http://testserver')


In [ ]:
@rt
def index(auth):
    u = users[auth]
    return P(f'Hello {u.email}!'), A('Log out', href='/logout')

@rt
def login(error: str=None):
    errmsg = P(error.replace('_', ' ').title(), style='color:red') if error else ''
    return Titled('Sign In', errmsg,
                Button('Sign in with Passkey', hx_post='/request_passkey_auth', target_id='scripts'),
                Hr(),
                Form(method='post', action='/send_magic_link')(
                    Input(name='email', type='email', placeholder='you@example.com'),
                    Button('Send Magic Link')),
                Div(id='scripts'))

@rt
def setup_passkey(): return Titled('Set Up Passkey',
                            P('Set up a passkey for faster logins next time?'),
                            Button('Register Passkey', hx_post='/request_passkey_reg', target_id='scripts'),
                            Form(Button('Skip'), method='post', action='/skip_passkey_reg'),
                            Div(id='scripts'))

In [ ]:
cli = TestClient(app)

In [ ]:
r = cli.get('/', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

r = cli.get('/login')
test_eq(r.status_code, 200)

In [ ]:
r = cli.post('/send_magic_link', data={'email': 'expired@test.com'})
token = [k for k in mk.magiclink_db.keys()][-1]
mk.magiclink_db[token]['timestamp'] = time.time() - 3601  # older than 1hr
r = cli.get(f'/verify_magiclink?token={token}', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login?error=invalid_link')


In [ ]:
r = cli.post('/send_magic_link', data={'email': 'test@example.com'})
test_eq(r.status_code, 200)
token = list(mk.magiclink_db.keys())[-1]

r = cli.get(f'/verify_magiclink?token={token}', follow_redirects=False)
test_eq(r.status_code, 303)
test(r.headers['location'], '/setup_passkey', operator.contains)

In [ ]:
r = cli.post('/skip_passkey_reg', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/')

r = cli.get('/')
test_eq(r.status_code, 200)
test(r.text, 'test@example.com', operator.contains)

In [ ]:
r = cli.get('/logout', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

r = cli.get('/', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

Lets add some tests for the unhappy paths

In [ ]:
r = cli.get(f'/verify_magiclink?token={token}', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login?error=invalid_link')

r = cli.get('/verify_magiclink?token=bogus', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login?error=invalid_link')


In [ ]:
r = cli.post('/verify_passkey_auth', data={'response': '{}', 'id': 'fake', 'rawId': 'fake', 'type': 'public-key'}, follow_redirects=False)
test_eq(r.status_code, 200)
test_eq(r.headers['hx-redirect'], '/login?error=no_challenge')

In [ ]:
r = cli.post('/request_passkey_auth', headers={'hx-request': 'true'})
r = cli.post('/verify_passkey_auth', data={'response': '{}', 'id': 'fake', 'rawId': 'fake', 'type': 'public-key'}, follow_redirects=False)
test_eq(r.status_code, 200)
test_eq(r.headers['hx-redirect'], '/login?error=passkey_not_found')

In [ ]:
r = cli.post('/finish_passkey_reg', data={'response': '{}', 'id': 'fake', 'rawId': 'fake', 'type': 'public-key'}, follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login?error=no_pending_reg')

## Export -

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()